The marketing team at Globalmart wants to analyze the purchasing patterns of customers to identify trends and opportunities for targeted marketing campaigns. Specifically, they are interested in understanding the time gap between consecutive purchases made by customers.


In [1]:
import pandas as pd
import numpy as np

In [2]:
order = pd.read_excel("https://cdn.enqurious.com/others/bf360f72-b500-4643-bc73-68da78ad8a84__globalmart.xlsx", sheet_name='orders')
order.columns

Index(['order_id', 'customer_id', 'ship_mode', 'vendor_id', 'order_status',
       'order_purchase_date', 'order_approved_at',
       'order_delivered_carrier_date', 'order_delivered_customer_date',
       'order_estimated_delivery_date'],
      dtype='object')

### Step 1: We have to create the base data 

In [3]:
# Convert the 'order_purchase_date' column to datetime format for further processing
order['order_date'] = pd.to_datetime(order['order_purchase_date'])

# Select relevant columns: 'order_id', 'customer_id', and the newly created 'order_date'
df = order[['order_id', 'customer_id', 'order_date']]

# Drop any rows where 'customer_id' is missing to ensure data integrity
df = df.dropna(subset=['customer_id'])

# Display the first few rows of the DataFrame to verify the changes
df.head()


,order_id,customer_id,order_date
1,CA-2014-101476,SD-20485,2017-01-23 13:40:00
3,CA-2014-101931,TS-21370,2017-10-22 14:25:00
4,CA-2014-103058,AG-10270,2018-01-30 11:03:00
5,CA-2014-103100,AB-10105,2018-08-15 09:38:00
6,CA-2014-103317,DM-13525,2018-08-06 08:44:00


### Step 2: We have to create previous/next order date by window function and calulate Days_since_last_order

In [4]:
# Sort the DataFrame by 'customer_id' and 'order_date' to organize orders chronologically for each customer
df = df.sort_values(by=['customer_id', 'order_date'])

# Create a 'previous_order_date' column that shows the previous order date for each customer using the shift function
df['previous_order_date'] = df.groupby('customer_id')['order_date'].shift(1)

# Create a 'next_order_date' column that shows the next order date for each customer using the shift function (shifted backward)
#df['next_order_date'] = df.groupby('customer_id')['order_date'].shift(-1)

# Calculate the number of days since the last order for each customer
df['Days_since_last_order'] = (df['order_date'] - df['previous_order_date']).dt.days

# Display the DataFrame to verify the changes
df


,order_id,customer_id,order_date,previous_order_date,Days_since_last_order
2297,CA-2014-138100,AA-10315,2017-01-17 14:57:00,NaT,NaN
3098,CA-2016-103982,AA-10315,2017-06-22 15:53:00,2017-01-17 14:57:00,156.0
4259,CA-2017-147039,AA-10315,2017-09-13 18:49:00,2017-06-22 15:53:00,83.0
159,CA-2015-121391,AA-10315,2018-01-15 09:31:00,2017-09-13 18:49:00,123.0
2235,CA-2014-128055,AA-10315,2018-06-12 15:48:00,2018-01-15 09:31:00,148.0
...,...,...,...,...,...
911,CA-2016-152471,ZD-21925,2018-03-12 14:34:00,NaT,NaN
3677,CA-2016-167682,ZD-21925,2018-03-29 08:51:00,2018-03-12 14:34:00,16.0
1151,CA-2014-143336,ZD-21925,2018-05-04 11:19:00,2018-03-29 08:51:00,36.0
4796,US-2016-147991,ZD-21925,2018-05-21 17:00:00,2018-05-04 11:19:00,17.0


### Step 3: We have to calulate the Cumulative Order Count for each customer

In [5]:
# Sort the DataFrame by 'order_date' to ensure that orders are in chronological order for each customer
df = df.sort_values(by='order_date')

# Calculate the Cumulative Order Count for each customer
# The 'cumcount()' function returns the cumulative number of orders per customer, starting from 0, so we add 1 to start from 1
df['Cumulative_order_count'] = df.groupby('customer_id').cumcount() + 1

# Display the updated DataFrame to verify the cumulative order count column
df


,order_id,customer_id,order_date,previous_order_date,Days_since_last_order,Cumulative_order_count
4130,CA-2017-135650,AC-10660,2016-10-03 22:31:00,NaT,NaN,1
2408,CA-2014-153087,TC-20980,2016-10-04 13:15:00,NaT,NaN,1
812,CA-2014-142965,SW-20245,2016-10-04 19:25:00,NaT,NaN,1
3314,CA-2016-126858,JM-15265,2016-10-04 19:41:00,NaT,NaN,1
1621,CA-2017-105445,BP-11095,2016-10-04 21:35:00,NaT,NaN,1
...,...,...,...,...,...,...
4352,CA-2017-155089,DB-12910,2018-08-28 08:18:00,2018-02-24 17:02:00,184.0,8
1031,US-2016-162026,JE-15745,2018-08-28 15:11:00,2018-07-11 19:44:00,47.0,13
4052,CA-2017-128335,JA-15970,2018-08-28 22:30:00,2018-04-09 13:36:00,141.0,8
3398,CA-2016-136686,RF-19840,2018-08-30 10:24:00,2018-07-26 18:54:00,34.0,8
